# Multi-Factor Equity Strategy

### A historical S&P 500 backtest using Value, Quality, Financial Strength and Momentum

I built this project to test a simple question: can a sector-relative multi-factor stock selection model outperform the S&P 500?

The first version of the backtest looked much stronger than I expected. That led me to audit the result rather than stop at the headline performance. The most important issue I found was the investment universe. Using today's S&P 500 members in earlier years introduces future information into the test. I therefore rebuilt the universe using historical index membership and reran the same model without changing the factor weights.

The corrected model did not beat SPY over the full sample. That result is the main finding of the project.

**Sample:** April 2020 to April 2026  
**Portfolio:** Top 20 stocks, equal weighted, annual rebalance  
**Benchmark:** SPY  
**Factors:** Value, Quality, Financial Strength, Momentum  
**Main data:** SEC EDGAR/XBRL and historical market prices


## 1. Setup

The notebook uses pandas and NumPy for the data work, yfinance for market prices and selected reference data, and SEC company facts for historical fundamentals.


In [ ]:
!pip install yfinance -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import requests
import time
from io import StringIO

print("Project setup works!")

## 2. Benchmark

SPY is used as the benchmark. I keep the benchmark construction separate from the stock-selection model so that the portfolio and index returns can be compared over the same holding periods.


In [ ]:
spy = yf.Ticker("SPY")

spy_data = spy.history(
    start="2015-01-01"
)

spy_data["Daily Return"] = spy_data["Close"].pct_change()
spy_data["Growth of $1"] = (
    1 + spy_data["Daily Return"].fillna(0)
).cumprod()

spy_data.head()

## 3. Starting investment universe

I first load the current S&P 500 constituent table and standardize ticker symbols and sectors. This current universe is useful for building and testing the pipeline, but it is not sufficient for a historical backtest. The historical-membership correction is introduced later in the notebook.


In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers_wiki = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers_wiki)
tables = pd.read_html(StringIO(response.text))
sp500 = tables[0]

sp500 = sp500[
    ["Symbol", "Security", "GICS Sector"]
].copy()

sp500.columns = [
    "Ticker",
    "Company",
    "Sector"
]

tickers = (
    sp500["Ticker"]
    .str.replace(".", "-", regex=False)
    .tolist()
)

print("Companies:", len(sp500))
print("Sectors:", sp500["Sector"].nunique())
sp500.head()

## 4. Momentum

Momentum is based on trailing price performance before each rebalance date. Scores are ranked within sector so that the model does not simply favor sectors that happened to have stronger recent performance.


In [ ]:
prices = yf.download(
    tickers,
    period="13mo",
    auto_adjust=True,
    progress=False
)["Close"]

momentum_12m = (
    prices.iloc[-1] / prices.iloc[-252] - 1
).dropna()

momentum_table = momentum_12m.reset_index()
momentum_table.columns = ["Ticker", "Momentum_12M"]

momentum_table["Momentum_Score"] = (
    momentum_table["Momentum_12M"]
    .rank(pct=True) * 100
)

momentum_table = momentum_table.sort_values(
    "Momentum_Score",
    ascending=False
)

momentum_table.head(20)

## 5. Value

The Value factor uses trailing P/E. Lower P/E receives a higher score within sector. Historical P/E is reconstructed later using point-in-time diluted EPS and split adjustments.


In [ ]:
fundamentals = []

for i, ticker in enumerate(tickers):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        fundamentals.append({
            "Ticker": ticker,
            "Forward_PE": info.get("forwardPE"),
            "EV_EBITDA": info.get("enterpriseToEbitda")
        })

    except Exception:
        fundamentals.append({
            "Ticker": ticker,
            "Forward_PE": np.nan,
            "EV_EBITDA": np.nan
        })

    if (i + 1) % 50 == 0:
        print(f"Completed {i + 1} / {len(tickers)}")

fundamentals = pd.DataFrame(fundamentals)

value_data = fundamentals.copy()

value_data.loc[
    value_data["Forward_PE"] <= 0,
    "Forward_PE"
] = np.nan

value_data.loc[
    value_data["EV_EBITDA"] <= 0,
    "EV_EBITDA"
] = np.nan

value_data["PE_Score"] = (
    value_data["Forward_PE"]
    .rank(pct=True, ascending=False) * 100
)

value_data["EV_EBITDA_Score"] = (
    value_data["EV_EBITDA"]
    .rank(pct=True, ascending=False) * 100
)

value_data["Value_Score"] = value_data[
    ["PE_Score", "EV_EBITDA_Score"]
].mean(axis=1, skipna=True)

value_data.sort_values(
    "Value_Score",
    ascending=False
).head(20)

## 6. Quality and Financial Strength

Quality is based on profitability measures, including operating margin and ROE. Financial Strength uses debt-to-equity, with lower leverage receiving a higher score. Financial companies are excluded because these accounting ratios are not directly comparable with operating companies.


In [ ]:
quality_strength_data = []

for i, ticker in enumerate(tickers):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        quality_strength_data.append({
            "Ticker": ticker,
            "ROE": info.get("returnOnEquity"),
            "Operating_Margin": info.get("operatingMargins"),
            "Debt_Equity": info.get("debtToEquity")
        })

    except Exception:
        quality_strength_data.append({
            "Ticker": ticker,
            "ROE": np.nan,
            "Operating_Margin": np.nan,
            "Debt_Equity": np.nan
        })

    if (i + 1) % 50 == 0:
        print(f"Completed {i + 1} / {len(tickers)}")

quality_strength_data = pd.DataFrame(quality_strength_data)

quality_strength_data["ROE_Score"] = (
    quality_strength_data["ROE"]
    .rank(pct=True, ascending=True) * 100
)

quality_strength_data["Operating_Margin_Score"] = (
    quality_strength_data["Operating_Margin"]
    .rank(pct=True, ascending=True) * 100
)

quality_strength_data["Quality_Score"] = (
    quality_strength_data[
        ["ROE_Score", "Operating_Margin_Score"]
    ]
    .mean(axis=1, skipna=True)
)

quality_strength_data["Strength_Score"] = (
    quality_strength_data["Debt_Equity"]
    .rank(pct=True, ascending=False) * 100
)

quality_strength_data.head()

## 7. Three-factor model

The first model combines Quality, Financial Strength and Momentum. Each signal is converted into a sector-relative percentile score and then combined into a single ranking.


In [ ]:
master = sp500.copy()

master = master.merge(
    momentum_table[
        ["Ticker", "Momentum_12M", "Momentum_Score"]
    ],
    on="Ticker",
    how="left"
)

master = master.merge(
    value_data[
        ["Ticker", "Forward_PE", "EV_EBITDA", "Value_Score"]
    ],
    on="Ticker",
    how="left"
)

master = master.merge(
    quality_strength_data[
        [
            "Ticker",
            "ROE",
            "Operating_Margin",
            "Debt_Equity",
            "Quality_Score",
            "Strength_Score"
        ]
    ],
    on="Ticker",
    how="left"
)

master["Multi_Factor_Score"] = (
    0.25 * master["Value_Score"] +
    0.25 * master["Quality_Score"] +
    0.25 * master["Strength_Score"] +
    0.25 * master["Momentum_Score"]
)

eligible = master.dropna(
    subset=[
        "Momentum_Score",
        "Value_Score",
        "Quality_Score",
        "Strength_Score",
        "Multi_Factor_Score"
    ]
).copy()

ranking = eligible.sort_values(
    "Multi_Factor_Score",
    ascending=False
)

ranking[
    [
        "Ticker",
        "Company",
        "Sector",
        "Value_Score",
        "Quality_Score",
        "Strength_Score",
        "Momentum_Score",
        "Multi_Factor_Score"
    ]
].head(20)

## 8. Four-factor model

The second model adds Value to the original three factors. I keep the weighting simple and transparent rather than optimizing weights on the same sample that is later used to judge performance.


In [ ]:
sector_map_df = sp500[["Ticker", "Sector"]].copy()

value_v2 = value_data.merge(
    sector_map_df,
    on="Ticker",
    how="left"
)

quality_v2 = quality_strength_data.merge(
    sector_map_df,
    on="Ticker",
    how="left"
)

value_v2["PE_Score_Sector"] = (
    value_v2
    .groupby("Sector")["Forward_PE"]
    .rank(pct=True, ascending=False) * 100
)

value_v2["EV_EBITDA_Score_Sector"] = (
    value_v2
    .groupby("Sector")["EV_EBITDA"]
    .rank(pct=True, ascending=False) * 100
)

value_v2["Value_Score_V2"] = value_v2[
    ["PE_Score_Sector", "EV_EBITDA_Score_Sector"]
].mean(axis=1, skipna=True)

quality_v2["ROE_Score_Sector"] = (
    quality_v2
    .groupby("Sector")["ROE"]
    .rank(pct=True, ascending=True) * 100
)

quality_v2["Operating_Margin_Score_Sector"] = (
    quality_v2
    .groupby("Sector")["Operating_Margin"]
    .rank(pct=True, ascending=True) * 100
)

quality_v2["Quality_Score_V2"] = quality_v2[
    ["ROE_Score_Sector", "Operating_Margin_Score_Sector"]
].mean(axis=1, skipna=True)

quality_v2["Strength_Score_V2"] = (
    quality_v2
    .groupby("Sector")["Debt_Equity"]
    .rank(pct=True, ascending=False) * 100
)

master_v2 = sp500.copy()

master_v2 = master_v2.merge(
    momentum_table[
        ["Ticker", "Momentum_12M", "Momentum_Score"]
    ],
    on="Ticker",
    how="left"
)

master_v2 = master_v2.merge(
    value_v2[["Ticker", "Value_Score_V2"]],
    on="Ticker",
    how="left"
)

master_v2 = master_v2.merge(
    quality_v2[
        ["Ticker", "Quality_Score_V2", "Strength_Score_V2"]
    ],
    on="Ticker",
    how="left"
)

master_v2["Multi_Factor_Score_V2"] = (
    0.25 * master_v2["Value_Score_V2"] +
    0.25 * master_v2["Quality_Score_V2"] +
    0.25 * master_v2["Strength_Score_V2"] +
    0.25 * master_v2["Momentum_Score"]
)

ranking_v2 = master_v2.dropna(
    subset=[
        "Value_Score_V2",
        "Quality_Score_V2",
        "Strength_Score_V2",
        "Momentum_Score",
        "Multi_Factor_Score_V2"
    ]
).sort_values(
    "Multi_Factor_Score_V2",
    ascending=False
)

ranking_v2.head(20)

## 9. Historical prices

Historical adjusted prices are downloaded for the backtest period. These prices are used for momentum, forward portfolio returns and the SPY benchmark.


In [ ]:
backtest_prices = yf.download(
    tickers + ["SPY"],
    start="2019-01-01",
    auto_adjust=True,
    progress=False
)["Close"]

raw_price_data = yf.download(
    tickers + ["SPY"],
    start="2019-01-01",
    auto_adjust=False,
    progress=False
)

raw_close = raw_price_data["Close"]

print("Adjusted prices:", backtest_prices.shape)
print("Raw closes:", raw_close.shape)

## 10. SEC EDGAR setup

Historical accounting data comes from SEC company facts. The request header needs a contact email, as required by the SEC's fair-access guidance.

Before running this notebook, replace `your-email@example.com` with your own contact email.


In [ ]:
SEC_EMAIL = "your-email@example.com"

headers = {
    "User-Agent": f"MultiFactorResearch {SEC_EMAIL}"
}

ticker_url = "https://www.sec.gov/files/company_tickers.json"

sec_tickers = requests.get(
    ticker_url,
    headers=headers
).json()

sec_tickers_df = pd.DataFrame.from_dict(
    sec_tickers,
    orient="index"
)

sec_tickers_df.head()

## 11. SEC helper functions

These functions standardize the SEC data, select filings available by the relevant date, calculate trailing-twelve-month values and retrieve balance-sheet observations without intentionally using later filings.


In [ ]:
def get_sec_concept(company_facts, concept):

    facts = company_facts["facts"]["us-gaap"]

    if concept not in facts:
        return None

    units = facts[concept]["units"]

    if "USD" not in units:
        return None

    df = pd.DataFrame(units["USD"])

    df["filed"] = pd.to_datetime(df["filed"])
    df["end"] = pd.to_datetime(df["end"])

    if "start" in df.columns:
        df["start"] = pd.to_datetime(df["start"])

    df = df[
        df["form"].isin(["10-K", "10-Q"])
    ].copy()

    return df

In [ ]:
def get_sec_shares(company_facts, concept):

    facts = company_facts["facts"]["us-gaap"]

    if concept not in facts:
        return None

    units = facts[concept]["units"]

    if "shares" not in units:
        return None

    df = pd.DataFrame(units["shares"])

    df["filed"] = pd.to_datetime(df["filed"])
    df["end"] = pd.to_datetime(df["end"])

    return df

In [ ]:
def calculate_ttm(df, as_of_date):

    as_of_date = pd.Timestamp(as_of_date)

    known = df[
        df["filed"] <= as_of_date
    ].copy()

    if known.empty:
        return np.nan

    known["duration_days"] = (
        known["end"] - known["start"]
    ).dt.days

    annual = known[
        (known["form"] == "10-K") &
        (known["fp"] == "FY")
    ].copy()

    if annual.empty:
        return np.nan

    latest_annual_filing = annual["filed"].max()

    annual_candidates = annual[
        annual["filed"] == latest_annual_filing
    ].copy()

    latest_fy = annual_candidates.sort_values(
        ["end", "duration_days"]
    ).iloc[-1]

    fy_value = latest_fy["val"]
    fy_end = latest_fy["end"]

    interim = known[
        (known["form"] == "10-Q") &
        (known["end"] > fy_end)
    ].copy()

    if interim.empty:
        return fy_value

    latest_interim_filing = interim["filed"].max()

    current_candidates = interim[
        interim["filed"] == latest_interim_filing
    ].copy()

    current_ytd_row = current_candidates.sort_values(
        "duration_days"
    ).iloc[-1]

    current_ytd = current_ytd_row["val"]
    current_duration = current_ytd_row["duration_days"]
    current_start = current_ytd_row["start"]
    current_end = current_ytd_row["end"]

    target_start = current_start - pd.DateOffset(years=1)
    target_end = current_end - pd.DateOffset(years=1)

    prior_candidates = known[
        known["end"] < current_start
    ].copy()

    if prior_candidates.empty:
        return np.nan

    prior_candidates["start_diff"] = (
        prior_candidates["start"] - target_start
    ).abs().dt.days

    prior_candidates["end_diff"] = (
        prior_candidates["end"] - target_end
    ).abs().dt.days

    prior_candidates["duration_diff"] = (
        prior_candidates["duration_days"] -
        current_duration
    ).abs()

    prior_candidates["match_score"] = (
        prior_candidates["start_diff"] +
        prior_candidates["end_diff"] +
        prior_candidates["duration_diff"]
    )

    prior_ytd_row = prior_candidates.sort_values(
        ["match_score", "filed"],
        ascending=[True, False]
    ).iloc[0]

    prior_ytd = prior_ytd_row["val"]

    return fy_value + current_ytd - prior_ytd

In [ ]:
def get_latest_balance_value(df, as_of_date):

    as_of_date = pd.Timestamp(as_of_date)

    known = df[
        df["filed"] <= as_of_date
    ].copy()

    if known.empty:
        return np.nan

    latest_end = known["end"].max()

    latest = known[
        known["end"] == latest_end
    ].copy()

    latest = latest.sort_values("filed")

    return latest.iloc[-1]["val"]

In [ ]:
def get_latest_shares(df, as_of_date):

    as_of_date = pd.Timestamp(as_of_date)

    known = df[
        df["filed"] <= as_of_date
    ].copy()

    if known.empty:
        return np.nan

    latest_end = known["end"].max()

    latest = known[
        known["end"] == latest_end
    ].sort_values("filed")

    return latest.iloc[-1]["val"]

## 12. Accounting concept fallbacks

XBRL tags are not perfectly consistent across issuers. I therefore use a small ordered set of concept fallbacks for revenue and other fundamentals rather than assuming every company reports under the same tag.


In [ ]:
REVENUE_CONCEPTS = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues",
    "SalesRevenueNet",
    "SalesRevenueGoodsNet"
]

SHARES_CONCEPTS = [
    "CommonStockSharesOutstanding",
    "EntityCommonStockSharesOutstanding"
]

EPS_CONCEPTS = [
    "EarningsPerShareDiluted",
    "EarningsPerShareBasicAndDiluted",
    "EarningsPerShareBasic"
]

In [ ]:
def get_first_available_concept(
    company_facts,
    concepts,
    unit="USD",
    namespaces=("us-gaap", "dei")
):

    for namespace in namespaces:

        if namespace not in company_facts["facts"]:
            continue

        facts = company_facts["facts"][namespace]

        for concept in concepts:

            if concept not in facts:
                continue

            units = facts[concept]["units"]

            if unit not in units:
                continue

            df = pd.DataFrame(units[unit])

            df["filed"] = pd.to_datetime(df["filed"])
            df["end"] = pd.to_datetime(df["end"])

            if "start" in df.columns:
                df["start"] = pd.to_datetime(df["start"])

            if "form" in df.columns:
                df = df[
                    df["form"].isin(["10-K", "10-Q"])
                ].copy()

            return df, concept, namespace

    return None, None, None

## 13. Historical fundamental snapshot

For each company and rebalance date, the snapshot function reconstructs the information needed by the factor model: revenue, operating income, net income, equity, debt, shares and the resulting ratios.


In [ ]:
def get_historical_snapshot(
    ticker,
    as_of_date,
    company_facts,
    prices
):

    as_of_date = pd.Timestamp(as_of_date)

    revenue, revenue_tag, revenue_namespace = (
        get_first_available_concept(
            company_facts,
            REVENUE_CONCEPTS,
            unit="USD"
        )
    )

    operating_income = get_sec_concept(
        company_facts,
        "OperatingIncomeLoss"
    )

    net_income = get_sec_concept(
        company_facts,
        "NetIncomeLoss"
    )

    ttm_revenue = (
        calculate_ttm(revenue, as_of_date)
        if revenue is not None
        else np.nan
    )

    ttm_operating_income = (
        calculate_ttm(operating_income, as_of_date)
        if operating_income is not None
        else np.nan
    )

    ttm_net_income = (
        calculate_ttm(net_income, as_of_date)
        if net_income is not None
        else np.nan
    )

    equity_df = get_sec_concept(
        company_facts,
        "StockholdersEquity"
    )

    equity = (
        get_latest_balance_value(
            equity_df,
            as_of_date
        )
        if equity_df is not None
        else np.nan
    )

    debt_current_df = get_sec_concept(
        company_facts,
        "LongTermDebtCurrent"
    )

    debt_noncurrent_df = get_sec_concept(
        company_facts,
        "LongTermDebtNoncurrent"
    )

    debt_current = (
        get_latest_balance_value(
            debt_current_df,
            as_of_date
        )
        if debt_current_df is not None
        else 0
    )

    debt_noncurrent = (
        get_latest_balance_value(
            debt_noncurrent_df,
            as_of_date
        )
        if debt_noncurrent_df is not None
        else 0
    )

    total_debt = debt_current + debt_noncurrent

    shares_df, shares_tag, shares_namespace = (
        get_first_available_concept(
            company_facts,
            SHARES_CONCEPTS,
            unit="shares"
        )
    )

    shares = (
        get_latest_shares(
            shares_df,
            as_of_date
        )
        if shares_df is not None
        else np.nan
    )

    price = prices.loc[
        as_of_date,
        ticker
    ]

    operating_margin = (
        ttm_operating_income / ttm_revenue
        if pd.notna(ttm_revenue)
        and pd.notna(ttm_operating_income)
        and ttm_revenue != 0
        else np.nan
    )

    roe = (
        ttm_net_income / equity
        if pd.notna(ttm_net_income)
        and pd.notna(equity)
        and equity != 0
        else np.nan
    )

    debt_equity = (
        total_debt / equity
        if pd.notna(equity)
        and equity != 0
        else np.nan
    )

    market_cap = (
        price * shares
        if pd.notna(price)
        and pd.notna(shares)
        else np.nan
    )

    trailing_pe = (
        market_cap / ttm_net_income
        if pd.notna(market_cap)
        and pd.notna(ttm_net_income)
        and ttm_net_income > 0
        else np.nan
    )

    return {
        "Ticker": ticker,
        "Date": as_of_date,
        "Price": price,
        "Shares": shares,
        "Market_Cap": market_cap,
        "TTM_Revenue": ttm_revenue,
        "TTM_Operating_Income": ttm_operating_income,
        "TTM_Net_Income": ttm_net_income,
        "Operating_Margin": operating_margin,
        "Equity": equity,
        "ROE": roe,
        "Debt": total_debt,
        "Debt_Equity": debt_equity,
        "Trailing_PE": trailing_pe,
        "Revenue_Tag": revenue_tag,
        "Revenue_Namespace": revenue_namespace,
        "Shares_Tag": shares_tag,
        "Shares_Namespace": shares_namespace
    }

## 14. SEC cache

SEC responses are cached during the session so the same company facts do not need to be downloaded repeatedly.


In [ ]:
sec_cache = {}

def get_company_facts(ticker):

    match = sec_tickers_df[
        sec_tickers_df["ticker"] == ticker
    ]

    if match.empty:
        return None

    cik = str(
        match["cik_str"].iloc[0]
    ).zfill(10)

    url = (
        "https://data.sec.gov/api/xbrl/companyfacts/"
        f"CIK{cik}.json"
    )

    response = requests.get(
        url,
        headers=headers
    )

    if response.status_code != 200:
        return None

    return response.json()

def get_cached_company_facts(ticker):

    if ticker in sec_cache:
        return sec_cache[ticker]

    facts = get_company_facts(ticker)
    sec_cache[ticker] = facts
    return facts

## 15. Initial historical universe

This section creates the first historical research universe from the available non-financial names. Later, I replace this with date-specific S&P 500 membership to address constituent-universe bias.


In [ ]:
historical_universe = sp500[
    sp500["Sector"] != "Financials"
].copy()

sector_map = (
    historical_universe
    .set_index("Ticker")["Sector"]
)

print("Historical universe:", len(historical_universe))
historical_universe["Sector"].value_counts()

## 16. Historical P/E and stock splits

Historical trailing P/E requires more care than simply dividing an old price by EPS. I reconstruct trailing diluted EPS from SEC filings and adjust for stock splits that occurred after the measurement date when necessary.


In [ ]:
def get_future_split_factor(
    ticker,
    as_of_date
):

    as_of_date = pd.Timestamp(as_of_date)

    stock = yf.Ticker(ticker)
    splits = stock.splits

    if splits.empty:
        return 1.0

    try:
        splits.index = splits.index.tz_localize(None)
    except TypeError:
        pass

    future_splits = splits[
        splits.index > as_of_date
    ]

    if future_splits.empty:
        return 1.0

    return future_splits.prod()

In [ ]:
def get_historical_ttm_eps(
    ticker,
    company_facts,
    as_of_date
):

    eps_df, eps_tag, eps_namespace = (
        get_first_available_concept(
            company_facts,
            EPS_CONCEPTS,
            unit="USD/shares"
        )
    )

    if eps_df is None:
        return np.nan

    try:
        ttm_eps = calculate_ttm(
            eps_df,
            as_of_date
        )
    except Exception:
        return np.nan

    if pd.isna(ttm_eps):
        return np.nan

    try:
        split_factor = get_future_split_factor(
            ticker,
            as_of_date
        )
    except Exception:
        split_factor = 1.0

    if pd.isna(split_factor) or split_factor <= 0:
        split_factor = 1.0

    return ttm_eps / split_factor

In [ ]:
def get_historical_pe(
    ticker,
    company_facts,
    as_of_date,
    prices
):

    as_of_date = pd.Timestamp(as_of_date)

    eps = get_historical_ttm_eps(
        ticker,
        company_facts,
        as_of_date
    )

    if pd.isna(eps) or eps <= 0:
        return np.nan

    try:
        price = prices.loc[
            as_of_date,
            ticker
        ]
    except Exception:
        return np.nan

    if pd.isna(price):
        return np.nan

    return price / eps

## 17. Frozen three-factor backtest

This function runs the original three-factor model for one annual holding period. The model is kept as a baseline rather than rewritten after seeing later results.


In [ ]:
def run_backtest_year(rebalance_date, top_n=20):

    rebalance_date = pd.Timestamp(rebalance_date)
    price_dates = backtest_prices.index

    start_date = price_dates[
        price_dates >= rebalance_date
    ][0]

    target_end = (
        rebalance_date
        + pd.DateOffset(years=1)
    )

    end_date = price_dates[
        price_dates >= target_end
    ][0]

    results = []
    errors = []

    for ticker in historical_universe["Ticker"]:

        try:
            facts = get_cached_company_facts(ticker)

            if facts is None:
                errors.append({
                    "Ticker": ticker,
                    "Error": "SEC facts unavailable"
                })
                continue

            snapshot = get_historical_snapshot(
                ticker,
                start_date,
                facts,
                raw_close
            )

            results.append(snapshot)

        except Exception as e:
            errors.append({
                "Ticker": ticker,
                "Error": str(e)
            })

    df = pd.DataFrame(results)

    if df.empty:
        raise ValueError(
            f"No valid company snapshots for {start_date}. "
            f"First errors: {errors[:5]}"
        )

    df["Sector"] = df["Ticker"].map(sector_map)

    target_past = (
        start_date
        - pd.DateOffset(years=1)
    )

    past_date = price_dates[
        price_dates <= target_past
    ][-1]

    momentum = (
        backtest_prices.loc[start_date]
        /
        backtest_prices.loc[past_date]
        - 1
    )

    df["Momentum"] = df["Ticker"].map(momentum)

    df["Operating_Margin_Clean"] = (
        df["Operating_Margin"]
        .where(
            df["Operating_Margin"]
            .between(-1, 1)
        )
    )

    df["ROE_Clean"] = (
        df["ROE"]
        .where(
            df["ROE"].between(-3, 3)
        )
    )

    df["Debt_Equity_Clean"] = (
        df["Debt_Equity"]
        .where(
            df["Debt_Equity"].between(0, 10)
        )
    )

    df["ROE_Score"] = (
        df
        .groupby("Sector")["ROE_Clean"]
        .rank(
            pct=True,
            ascending=True
        )
        * 100
    )

    df["Margin_Score"] = (
        df
        .groupby("Sector")["Operating_Margin_Clean"]
        .rank(
            pct=True,
            ascending=True
        )
        * 100
    )

    df["Quality_Score"] = (
        df[
            ["ROE_Score", "Margin_Score"]
        ]
        .mean(axis=1)
    )

    df["Strength_Score"] = (
        df
        .groupby("Sector")["Debt_Equity_Clean"]
        .rank(
            pct=True,
            ascending=False
        )
        * 100
    )

    df["Momentum_Score"] = (
        df
        .groupby("Sector")["Momentum"]
        .rank(
            pct=True,
            ascending=True
        )
        * 100
    )

    df["Factor_Count"] = (
        df[
            [
                "Quality_Score",
                "Strength_Score",
                "Momentum_Score"
            ]
        ]
        .notna()
        .sum(axis=1)
    )

    eligible = df[
        df["Factor_Count"] == 3
    ].copy()

    eligible["Multi_Factor_Score"] = (
        eligible[
            [
                "Quality_Score",
                "Strength_Score",
                "Momentum_Score"
            ]
        ]
        .mean(axis=1)
    )

    if (
        "GOOG" in eligible["Ticker"].values
        and
        "GOOGL" in eligible["Ticker"].values
    ):
        alphabet = eligible[
            eligible["Ticker"].isin(
                ["GOOG", "GOOGL"]
            )
        ]

        keep = alphabet.loc[
            alphabet["Multi_Factor_Score"].idxmax(),
            "Ticker"
        ]

        remove = (
            "GOOGL"
            if keep == "GOOG"
            else "GOOG"
        )

        eligible = eligible[
            eligible["Ticker"] != remove
        ]

    portfolio = (
        eligible
        .sort_values(
            "Multi_Factor_Score",
            ascending=False
        )
        .head(top_n)
        .copy()
    )

    portfolio_tickers = portfolio["Ticker"].tolist()

    forward_returns = (
        backtest_prices.loc[
            end_date,
            portfolio_tickers
        ]
        /
        backtest_prices.loc[
            start_date,
            portfolio_tickers
        ]
        - 1
    )

    portfolio["Forward_Return"] = (
        portfolio["Ticker"]
        .map(forward_returns)
    )

    portfolio_return = (
        portfolio["Forward_Return"]
        .mean()
    )

    spy_return = (
        backtest_prices.loc[
            end_date,
            "SPY"
        ]
        /
        backtest_prices.loc[
            start_date,
            "SPY"
        ]
        - 1
    )

    return {
        "Rebalance_Date": start_date,
        "End_Date": end_date,
        "Eligible_Stocks": len(eligible),
        "Portfolio_Return": portfolio_return,
        "SPY_Return": spy_return,
        "Excess_Return": portfolio_return - spy_return,
        "Portfolio": portfolio,
        "Errors": pd.DataFrame(errors)
    }

## 18. Baseline results

I run the three-factor model from 2020 through 2025 and compare each annual holding period with SPY.


In [ ]:
backtest_results = []

for date in [
    "2020-04-01",
    "2021-04-01",
    "2022-04-01",
    "2023-04-01",
    "2024-04-01",
    "2025-04-01"
]:

    print(f"Running {date}...")

    result = run_backtest_year(date)

    backtest_results.append({
        "Year": pd.Timestamp(date).year,
        "Start_Date": result["Rebalance_Date"],
        "End_Date": result["End_Date"],
        "Eligible_Stocks": result["Eligible_Stocks"],
        "Portfolio_Return": result["Portfolio_Return"],
        "SPY_Return": result["SPY_Return"],
        "Excess_Return": result["Excess_Return"]
    })

backtest_summary = pd.DataFrame(backtest_results)

backtest_summary.style.format({
    "Portfolio_Return": "{:.2%}",
    "SPY_Return": "{:.2%}",
    "Excess_Return": "{:.2%}"
})

### Baseline result

Across the six annual periods, the original three-factor strategy finished below SPY overall. This became the benchmark for testing whether adding Value improved the model.


In [ ]:
n_years = len(backtest_summary)

strategy_growth = (
    1 + backtest_summary["Portfolio_Return"]
).prod()

spy_growth = (
    1 + backtest_summary["SPY_Return"]
).prod()

strategy_cagr = (
    strategy_growth ** (1 / n_years)
    - 1
)

spy_cagr = (
    spy_growth ** (1 / n_years)
    - 1
)

strategy_vol = (
    backtest_summary["Portfolio_Return"]
    .std()
)

spy_vol = (
    backtest_summary["SPY_Return"]
    .std()
)

strategy_sharpe = (
    backtest_summary["Portfolio_Return"].mean()
    / strategy_vol
)

spy_sharpe = (
    backtest_summary["SPY_Return"].mean()
    / spy_vol
)

years_outperformed = (
    backtest_summary["Excess_Return"] > 0
).sum()

print(f"Strategy final value: ${strategy_growth:.3f}")
print(f"SPY final value:      ${spy_growth:.3f}")
print()
print(f"Strategy CAGR: {strategy_cagr:.2%}")
print(f"SPY CAGR:      {spy_cagr:.2%}")
print()
print(f"Strategy volatility: {strategy_vol:.2%}")
print(f"SPY volatility:      {spy_vol:.2%}")
print()
print(f"Strategy Sharpe: {strategy_sharpe:.2f}")
print(f"SPY Sharpe:      {spy_sharpe:.2f}")
print()
print(
    f"Outperformed SPY: "
    f"{years_outperformed}/{n_years} periods"
)

In [ ]:
growth_table = backtest_summary[
    [
        "Year",
        "Portfolio_Return",
        "SPY_Return"
    ]
].copy()

growth_table["Strategy_$1"] = (
    1 + growth_table["Portfolio_Return"]
).cumprod()

growth_table["SPY_$1"] = (
    1 + growth_table["SPY_Return"]
).cumprod()

growth_table

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    growth_table["Year"],
    growth_table["Strategy_$1"],
    marker="o",
    label="Strategy"
)
plt.plot(
    growth_table["Year"],
    growth_table["SPY_$1"],
    marker="o",
    label="SPY"
)
plt.title("Growth of $1: Historical 3-Factor Strategy vs SPY")
plt.xlabel("Rebalance Year")
plt.ylabel("Portfolio Value ($)")
plt.grid(True)
plt.legend()
plt.show()

## 19. Four-factor backtest

Value is added to create V2. The portfolio remains equal weighted and uses the same annual rebalance structure so that the comparison with V1 is straightforward.


In [ ]:
def run_backtest_year_v2(rebalance_date, top_n=20):

    rebalance_date = pd.Timestamp(rebalance_date)
    price_dates = backtest_prices.index

    start_date = price_dates[
        price_dates >= rebalance_date
    ][0]

    target_end = (
        rebalance_date
        + pd.DateOffset(years=1)
    )

    end_date = price_dates[
        price_dates >= target_end
    ][0]

    results = []
    errors = []

    for ticker in historical_universe["Ticker"]:

        try:
            facts = get_cached_company_facts(ticker)

            if facts is None:
                continue

            snapshot = get_historical_snapshot(
                ticker,
                start_date,
                facts,
                raw_close
            )

            snapshot["Historical_PE"] = get_historical_pe(
                ticker,
                facts,
                start_date,
                raw_close
            )

            results.append(snapshot)

        except Exception as e:
            errors.append({
                "Ticker": ticker,
                "Error": str(e)
            })

    df = pd.DataFrame(results)

    if df.empty:
        raise ValueError(
            f"No valid snapshots for {start_date}. "
            f"First errors: {errors[:5]}"
        )

    df["Sector"] = df["Ticker"].map(sector_map)

    target_past = (
        start_date
        - pd.DateOffset(years=1)
    )

    past_date = price_dates[
        price_dates <= target_past
    ][-1]

    momentum = (
        backtest_prices.loc[start_date]
        /
        backtest_prices.loc[past_date]
        - 1
    )

    df["Momentum"] = df["Ticker"].map(momentum)

    df["Operating_Margin_Clean"] = (
        df["Operating_Margin"]
        .where(
            df["Operating_Margin"].between(-1, 1)
        )
    )

    df["ROE_Clean"] = (
        df["ROE"]
        .where(
            df["ROE"].between(-3, 3)
        )
    )

    df["Debt_Equity_Clean"] = (
        df["Debt_Equity"]
        .where(
            df["Debt_Equity"].between(0, 10)
        )
    )

    df["PE_Clean"] = (
        df["Historical_PE"]
        .where(
            df["Historical_PE"].between(2, 100)
        )
    )

    df["Value_Score"] = (
        df
        .groupby("Sector")["PE_Clean"]
        .rank(
            pct=True,
            ascending=False
        )
        * 100
    )

    df["ROE_Score"] = (
        df
        .groupby("Sector")["ROE_Clean"]
        .rank(
            pct=True,
            ascending=True
        )
        * 100
    )

    df["Margin_Score"] = (
        df
        .groupby("Sector")["Operating_Margin_Clean"]
        .rank(
            pct=True,
            ascending=True
        )
        * 100
    )

    df["Quality_Score"] = (
        df[
            ["ROE_Score", "Margin_Score"]
        ]
        .mean(axis=1)
    )

    df["Strength_Score"] = (
        df
        .groupby("Sector")["Debt_Equity_Clean"]
        .rank(
            pct=True,
            ascending=False
        )
        * 100
    )

    df["Momentum_Score"] = (
        df
        .groupby("Sector")["Momentum"]
        .rank(
            pct=True,
            ascending=True
        )
        * 100
    )

    factor_columns = [
        "Value_Score",
        "Quality_Score",
        "Strength_Score",
        "Momentum_Score"
    ]

    df["Factor_Count"] = (
        df[factor_columns]
        .notna()
        .sum(axis=1)
    )

    eligible = df[
        df["Factor_Count"] == 4
    ].copy()

    eligible["Multi_Factor_Score"] = (
        eligible[factor_columns]
        .mean(axis=1)
    )

    if (
        "GOOG" in eligible["Ticker"].values
        and
        "GOOGL" in eligible["Ticker"].values
    ):
        alphabet = eligible[
            eligible["Ticker"].isin(
                ["GOOG", "GOOGL"]
            )
        ]

        keep = alphabet.loc[
            alphabet["Multi_Factor_Score"].idxmax(),
            "Ticker"
        ]

        remove = (
            "GOOGL"
            if keep == "GOOG"
            else "GOOG"
        )

        eligible = eligible[
            eligible["Ticker"] != remove
        ]

    portfolio = (
        eligible
        .sort_values(
            "Multi_Factor_Score",
            ascending=False
        )
        .head(top_n)
        .copy()
    )

    portfolio_tickers = portfolio["Ticker"].tolist()

    forward_returns = (
        backtest_prices.loc[
            end_date,
            portfolio_tickers
        ]
        /
        backtest_prices.loc[
            start_date,
            portfolio_tickers
        ]
        - 1
    )

    portfolio["Forward_Return"] = (
        portfolio["Ticker"]
        .map(forward_returns)
    )

    portfolio_return = (
        portfolio["Forward_Return"].mean()
    )

    spy_return = (
        backtest_prices.loc[end_date, "SPY"]
        /
        backtest_prices.loc[start_date, "SPY"]
        - 1
    )

    return {
        "Rebalance_Date": start_date,
        "End_Date": end_date,
        "Eligible_Stocks": len(eligible),
        "Portfolio_Return": portfolio_return,
        "SPY_Return": spy_return,
        "Excess_Return": portfolio_return - spy_return,
        "Portfolio": portfolio,
        "Universe": df,
        "Errors": pd.DataFrame(errors)
    }

In [ ]:
v2_results = []

for date in [
    "2020-04-01",
    "2021-04-01",
    "2022-04-01",
    "2023-04-01",
    "2024-04-01",
    "2025-04-01"
]:

    print(f"Running V2: {date}...")

    result = run_backtest_year_v2(date)

    v2_results.append({
        "Year": pd.Timestamp(date).year,
        "Start_Date": result["Rebalance_Date"],
        "End_Date": result["End_Date"],
        "Eligible_Stocks": result["Eligible_Stocks"],
        "V2_Return": result["Portfolio_Return"],
        "SPY_Return": result["SPY_Return"],
        "V2_Excess": result["Excess_Return"]
    })

print("\nV2 DONE!")

v2_summary = pd.DataFrame(v2_results)

v2_summary.style.format({
    "V2_Return": "{:.2%}",
    "SPY_Return": "{:.2%}",
    "V2_Excess": "{:.2%}"
})

In [ ]:
comparison = backtest_summary[
    ["Year", "Portfolio_Return", "SPY_Return"]
].copy()

comparison = comparison.rename(
    columns={
        "Portfolio_Return": "V1_Return"
    }
)

comparison = comparison.merge(
    v2_summary[
        ["Year", "V2_Return"]
    ],
    on="Year",
    how="left"
)

comparison["V1_Excess"] = (
    comparison["V1_Return"]
    - comparison["SPY_Return"]
)

comparison["V2_Excess"] = (
    comparison["V2_Return"]
    - comparison["SPY_Return"]
)

comparison[
    [
        "Year",
        "V1_Return",
        "V2_Return",
        "SPY_Return",
        "V1_Excess",
        "V2_Excess"
    ]
].style.format({
    "V1_Return": "{:.2%}",
    "V2_Return": "{:.2%}",
    "SPY_Return": "{:.2%}",
    "V1_Excess": "{:.2%}",
    "V2_Excess": "{:.2%}"
})

In [ ]:
robustness_years = [
    "2020-04-01",
    "2021-04-01",
    "2022-04-01",
    "2023-04-01",
    "2024-04-01",
    "2025-04-01"
]

robustness_results = []

for date in robustness_years:

    print(f"Running {date}...")

    result = run_backtest_year_v2(
        date,
        top_n=50
    )

    portfolio_ranked = (
        result["Portfolio"]
        .sort_values(
            "Multi_Factor_Score",
            ascending=False
        )
        .copy()
    )

    for top_n in [10, 20, 30]:

        subset = portfolio_ranked.head(top_n)

        portfolio_return = (
            subset["Forward_Return"].mean()
        )

        robustness_results.append({
            "Year": pd.Timestamp(date).year,
            "Top_N": top_n,
            "Portfolio_Return": portfolio_return,
            "SPY_Return": result["SPY_Return"],
            "Excess_Return": (
                portfolio_return
                - result["SPY_Return"]
            )
        })

print("\nDONE!")

In [ ]:
robustness_df = pd.DataFrame(
    robustness_results
)

robustness_df.style.format({
    "Portfolio_Return": "{:.2%}",
    "SPY_Return": "{:.2%}",
    "Excess_Return": "{:.2%}"
})

In [ ]:
robustness_summary = []

for top_n in [10, 20, 30]:

    temp = robustness_df[
        robustness_df["Top_N"] == top_n
    ].copy()

    final_value = (
        1 + temp["Portfolio_Return"]
    ).prod()

    cagr = (
        final_value ** (1 / len(temp))
        - 1
    )

    volatility = (
        temp["Portfolio_Return"].std()
    )

    sharpe = (
        temp["Portfolio_Return"].mean()
        / volatility
    )

    wins = (
        temp["Excess_Return"] > 0
    ).sum()

    robustness_summary.append({
        "Portfolio_Size": top_n,
        "Final_$1": final_value,
        "CAGR": cagr,
        "Volatility": volatility,
        "Sharpe": sharpe,
        "Periods_Beating_SPY": wins
    })

robustness_summary = pd.DataFrame(
    robustness_summary
)

robustness_summary.style.format({
    "Final_$1": "${:.3f}",
    "CAGR": "{:.2%}",
    "Volatility": "{:.2%}",
    "Sharpe": "{:.2f}"
})

In [ ]:
## 20. Portfolio-size robustness

I test Top 10, Top 20 and Top 30 portfolios. The purpose is not to select the best result after the fact. It is a check on whether the conclusion depends heavily on one arbitrary portfolio size.

The Top 20 specification remains the main model.


In [ ]:
## 21. Historical S&P 500 membership

This is the key robustness correction in the project.

The original model could select companies that are members of the S&P 500 today even if they were not members at the historical rebalance date. I use a historical constituent dataset to reconstruct the index membership for each annual formation date.


In [ ]:
### Historical membership check

The number of constituents is checked at each rebalance date before matching the historical membership list to available price data.


In [ ]:
### Current membership versus 2020 membership

This comparison shows why the correction matters. The S&P 500 changes meaningfully through time, so today's constituent list is not a valid substitute for the investable universe in 2020.


In [ ]:
### Expanding the historical price universe

I collect the union of tickers that appear across the historical membership dates so former constituents are not automatically lost just because they are absent from the current index.


In [ ]:
### Historical price panel

The expanded ticker list is used to create the price panel required for date-specific historical universes.


In [ ]:
### Price coverage

Coverage is measured explicitly for every rebalance date. This is important because a historical-membership backtest can still be biased if delisted or renamed securities systematically disappear from the available data.


In [ ]:
## 22. Date-specific investable universes

For each rebalance date I retain historical S&P 500 members with an available price on the formation date.


In [ ]:
### Sector mapping

The factor model ranks companies within sector, so historical constituents also need a sector classification. I first match names that are already covered by the existing sector map.


In [ ]:
### Missing historical sector mappings

Former constituents and renamed companies create gaps in the current sector table. I identify those names explicitly rather than dropping them silently.


In [ ]:
### Recovering sector classifications

Current S&P information recovers part of the missing mappings. The remaining historical names are handled separately below.


In [ ]:
### Historical sector recovery

For the remaining names, I retrieve an available sector classification and then standardize the labels to the GICS-style categories used in the rest of the project.


In [ ]:
### Standardizing sector names

Different data sources use slightly different sector labels. These are mapped into one consistent classification before the final historical universe is built.


In [ ]:
### Expanded non-financial sector map

Recovered historical classifications are combined with the original sector map. Financials remain excluded for accounting comparability.


In [ ]:
### Final historical coverage

This table reports how many historical index members have usable prices and non-financial sector mappings at each rebalance date.


In [ ]:
## 23. Corrected historical backtest

This function reruns the four-factor model using the reconstructed S&P 500 membership for each date.

Importantly, I do not change the factor definitions or weights after making the universe correction. This keeps the corrected test comparable with the original V2 result.


In [ ]:
## 24. Corrected V2 results

The corrected historical-universe model is run over the same six annual holding periods.


In [ ]:
### Original V2 versus corrected V2

This is the central comparison. The original V2 result is shown next to the historical-universe result and SPY so the effect of the universe correction is visible directly.


In [ ]:
### Performance summary

The original V2 produced a 24.32% CAGR. After reconstructing historical S&P 500 membership, CAGR fell to 14.63%, compared with 19.39% for SPY.

The corrected strategy also had higher volatility and a lower Sharpe ratio than SPY. In other words, the original result overstated the strength of the strategy.


In [ ]:
### Annual win rate

The corrected V2 beat SPY in 2 of the 6 annual periods.


In [ ]:
## 25. Factor diagnostics

I next test whether the individual factor scores are related to subsequent stock returns. Spearman information coefficients are calculated within each annual cross-section.

A positive IC means higher factor scores tended to be associated with higher forward returns.


In [ ]:
### Yearly IC summary

Value has the strongest average IC in the sample. Quality is close to neutral, while Financial Strength is negative in most years. Momentum is heavily affected by the 2020 reversal.


In [ ]:
## 26. Pooled factor sample

The annual cross-sections are combined into one stock-year dataset. This gives 1,759 observations for the pooled factor diagnostics.


In [ ]:
### Pooled information coefficients

The pooled results are small in magnitude. Value remains the strongest of the four signals, while Quality is close to zero. Momentum is approximately neutral once 2020 is excluded, and Financial Strength remains negative.


In [ ]:
## 27. Quintile analysis

For each factor, stocks are divided into five groups from the lowest score (Q1) to the highest score (Q5), and I compare their average subsequent returns.

Value is the clearest result. Average forward return rises from about 18.1% in Q1 to 28.7% in Q5. The other three factors do not show a similarly clean positive pattern.

## Conclusion

The initial four-factor strategy looked substantially better than SPY, but a large part of that apparent advantage disappeared after correcting the historical investment universe. The corrected model produced a 14.63% CAGR versus 19.39% for SPY, with higher volatility and a lower Sharpe ratio.

The factor tests still produced one useful result. Sector-relative Value showed the most consistent positive relationship with subsequent returns in this sample. Quality was mostly neutral, Financial Strength was weak, and Momentum was unstable around the 2020 market reversal.

For me, the main lesson from the project is methodological. A strong backtest is not enough on its own. The construction of the historical universe, the timing of accounting information, stock splits and missing data can materially change the result.

### Limitations

This is a research project, not an investable trading strategy. The sample contains only six annual holding periods. Historical price and sector coverage is not complete for every former constituent, transaction costs and taxes are excluded, and the Sharpe ratio is estimated from a small number of annual observations. The historical-universe version reduces constituent bias materially, but I do not describe it as perfectly survivorship-bias-free.
